# Problem 2: BMI-Stratified Optimal NIPT Timing

Estimate the earliest gestational week at which each BMI group reaches the target probability of sufficient fetal fraction, then assess sensitivity to measurement error.

## Data and reproducibility

The participant-level NIPT dataset is not distributed in this public repository. To reproduce the analysis, place an authorized copy at `data/nipt_data.xlsx` using the English schema documented in `data/README.md`.


## Setup


In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
DATA_PATH = REPO_ROOT / "data" / "nipt_data.xlsx"
OUTPUT_DIR = REPO_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "The authorized NIPT dataset is not included in this public repository. "
        "Place it at data/nipt_data.xlsx after reviewing data-use restrictions."
    )


## Core timing model


In [ ]:
# Font configuration
import os, glob, re
import numpy as np
import pandas as pd
import matplotlib, matplotlib.pyplot as plt
from matplotlib import font_manager as fm

def setup_plot_font(prefer=("DejaVu Sans",)):
    installed = {f.name for f in fm.fontManager.ttflist}
    for fam in prefer:
        if fam in installed:
            matplotlib.rcParams["font.family"] = fam
            matplotlib.rcParams["axes.unicode_minus"] = False
            matplotlib.rcParams["pdf.fonttype"] = 42
            matplotlib.rcParams["ps.fonttype"]  = 42
            print(f"[OK] Using font: {fam}")
            return
    candidates = [r"", r"", r""]
    for p in candidates:
        if os.path.exists(p):
            fm.fontManager.addfont(p)
            fam = fm.FontProperties(fname=p).get_name()
            matplotlib.rcParams["font.family"] = fam
            matplotlib.rcParams["axes.unicode_minus"] = False
            matplotlib.rcParams["pdf.fonttype"] = 42
            matplotlib.rcParams["ps.fonttype"]  = 42
            print(f"[OK] Registered font file: {p} -> {fam}")
            return
    print("[WARN] No preferred font found; using the default font")

setup_plot_font()


def safe_read_excel(candidates=None, sheet_candidates=None, engine="openpyxl"):
    cand = (candidates or []) + [r"nipt_data.xlsx", r"nipt_data.xlsx"]
    cand += glob.glob("**/nipt_data.xlsx", recursive=True)
    cand += glob.glob("**/nipt_data.xlsx", recursive=True)
    fpath = None
    for p in cand:
        if p and os.path.exists(p):
            fpath = p; break
    if fpath is None:
        raise FileNotFoundError("Data file not found")
    xls = pd.ExcelFile(fpath, engine=engine)
    pick = xls.sheet_names[0]
    print(f"[OK] Loaded: {os.path.abspath(fpath)} | Sheet: {pick}")
    return pd.read_excel(fpath, sheet_name=pick)


def find_col(cols, keys):
    cols = [str(c).strip() for c in cols]
    for k in keys:
        for c in cols:
            if k in c:
                return c
    return None


def parse_week(x):
    if pd.isna(x): return np.nan
    s = str(x).strip()
    m = re.findall(r"(\d+)\s*(?:w|week)\s*\+?\s*(\d+)?", s, flags=re.I)
    if m:
        w = int(m[0][0]); d = int(m[0][1]) if m[0][1] not in [None,""] else 0
        return w + d/7.0
    try: return float(s)
    except: return np.nan


try:
    df
except NameError:
    raw = safe_read_excel()
    col_bmi   = find_col(raw.columns, ["maternal_bmi","BMI"])
    col_ga    = find_col(raw.columns, ["gestational_age","gestational_age"])
    col_yconc = find_col(raw.columns, ["y_chromosome_fraction","y_chromosome_fraction"])
    df = raw[[col_bmi,col_ga,col_yconc]].copy()
    df[col_bmi]   = pd.to_numeric(df[col_bmi], errors="coerce")
    df[col_ga]    = df[col_ga].apply(parse_week)
    df[col_yconc] = pd.to_numeric(df[col_yconc], errors="coerce")
    df = df.dropna().reset_index(drop=True)

    weeks = np.arange(10.0, 26.0+1e-9, 0.1)
    TARGET_COVER = 0.60


    from sklearn.ensemble import RandomForestRegressor
    reg = RandomForestRegressor(n_estimators=300, random_state=42)
    reg.fit(df[[col_ga,col_bmi]].astype(float), df[col_yconc].astype(float))
    yhat = reg.predict(df[[col_ga,col_bmi]])
    sigma = float(np.std(df[col_yconc] - yhat, ddof=1))
    Z = 1.645
    CV = 0.10


    def gstar_array(bmis, cv=CV, sigma_scale=1.0):
        bmis = np.asarray(bmis, dtype=float)
        T = len(weeks); M = len(bmis)
        GA  = np.tile(weeks[:,None], (1,M))
        BMI = np.tile(bmis[None,:], (T,1))
        Xp  = pd.DataFrame({col_ga: GA.ravel(), col_bmi: BMI.ravel()})
        mu  = reg.predict(Xp).reshape(T,M)
        lo  = mu - Z*sigma*sigma_scale
        thr = 0.04*(1+cv)
        mask = lo >= thr
        ok   = mask.any(axis=0)
        idx  = np.where(ok, mask.argmax(axis=0), -1)
        return np.where(ok, weeks[idx], np.nan)


    def coverage_curve(gs):
        return np.array([float(np.nanmean(gs <= t)) for t in weeks])

    # BMI group
    bins   = [24,28,30,32,34,36,np.inf]
    labels = ["24–28","28–30","30–32","32–34","34–36","≥36"]
    df["bmi_group"] = pd.cut(df[col_bmi], bins=bins, labels=labels, right=False)
    label_order = [lab for lab in labels if lab in df["bmi_group"].dropna().unique()]


def build_plot_curves_if_needed():
    pc = {}
    for g in label_order:
        sub = df[df["bmi_group"]==g]
        if len(sub) < 5:
            continue
        bmis = sub[col_bmi].values
        gs   = gstar_array(bmis)
        cov  = coverage_curve(gs)
        idx  = np.where(cov >= TARGET_COVER)[0]
        t_cov = weeks[idx[0]] if len(idx) else weeks[-1]
        cov_at= cov[idx[0]] if len(idx) else cov[-1]
        pc[g] = (weeks, cov, t_cov, cov_at)
    return pc

plot_curves = build_plot_curves_if_needed()


palette = ["#1f77b4","#ff7f0e","#2ca02c","#9467bd","#d62728","#17becf"]


plt.figure(figsize=(9.5,5.6), facecolor="white")
ax = plt.gca(); ax.set_facecolor("white")


left, right = float(np.min(list(plot_curves.values())[0][0])) if plot_curves else 10.0, float(np.max(list(plot_curves.values())[0][0])) if plot_curves else 26.0
ax.axvspan(max(left,10), min(right,12), facecolor="#CCE5FF", alpha=0.35, label="<=12 weeks: low risk")
ax.axvspan(max(left,13), min(right,27), facecolor="#FFF3CD", alpha=0.45, label="13-27 weeks: high risk")
if right >= 28:
    ax.axvspan(28, right, facecolor="#F8D7DA", alpha=0.45, label=">=28 weeks: very high risk")


ax.axhline(TARGET_COVER, linewidth=1.8, color="#444", linestyle="--")
ax.text(right+0.05, TARGET_COVER, f"{int(TARGET_COVER*100)}% target", va="center", ha="left", fontsize=10, color="#444")


for i,(g,(w,cov,t_star,cov_star)) in enumerate(plot_curves.items()):
    c = palette[i % len(palette)]
    plt.step(w, cov, where="post", color=c, linewidth=2.6, label=g)
    plt.scatter([t_star],[cov_star], s=48, color=c, edgecolor="white", zorder=5)
    plt.annotate(f"{t_star:.1f}week · {cov_star:.0%}",
                 xy=(t_star, cov_star), xytext=(6,8), textcoords="offset points",
                 fontsize=10, color=c,
                 bbox=dict(boxstyle="round,pad=0.18", fc="white", ec=c, lw=0.8, alpha=0.9))

plt.xlim(weeks[0], weeks[-1]); plt.ylim(0,1.02)
plt.xlabel("gestational_age")
plt.ylabel("Threshold coverage  P(g*≤t)")
plt.title("Coverage curves by BMI group")
handles, labels_ = ax.get_legend_handles_labels()
plt.legend(handles, labels_, ncol=3, frameon=False, loc="lower right", fontsize=9)
for spine in ["top","right"]:
    ax.spines[spine].set_visible(False)
ax.spines["left"].set_alpha(0.6)
ax.spines["bottom"].set_alpha(0.6)
ax.tick_params(axis="both", labelsize=10)
plt.tight_layout()
plt.show()


rec_df = pd.DataFrame({
    "bmi_group": list(plot_curves.keys()),
    "Recommended week": [v[2] for v in plot_curves.values()],
    "coverage": [v[3] for v in plot_curves.values()],
}).sort_values("Recommended week")

plt.figure(figsize=(7.2,4.2), facecolor="white")
ax = plt.gca(); ax.set_facecolor("white")
bars = ax.barh(rec_df["bmi_group"], rec_df["Recommended week"], color="#6aaed6")
for y,(t,cov) in enumerate(zip(rec_df["Recommended week"], rec_df["coverage"])):
    ax.text(t+0.05, y, f"{t:.1f}week · {cov:.0%}", va="center", fontsize=10)
ax.set_xlabel("Recommended week")
ax.set_title("Recommended week summary by BMI group")
for spine in ["top","right"]:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()


## Export summary tables


In [ ]:


import os
import numpy as np
import pandas as pd


def _build_plot_curves_if_needed():
    try:
        return plot_curves
    except NameError:
        pass

    if "df" not in globals():
        raise RuntimeError("Missing df; load the data and create BMI groups first")
    if "gstar_array" not in globals() or "coverage_curve" not in globals():
        raise RuntimeError("Missing gstar_array or coverage_curve")
    if "weeks" not in globals() or "TARGET_COVER" not in globals():
        raise RuntimeError("Missing weeks or TARGET_COVER")


    labs = globals().get("label_order")
    if labs is None:
        if "bmi_group" not in df.columns:
            raise RuntimeError("Missing bmi_group; create BMI groups first")
        labs = list(df["bmi_group"].dropna().astype(str).unique())

    pc = {}
    for g in labs:
        sub = df[df["bmi_group"].astype(str) == g]
        if len(sub) < 5:
            continue
        bmis = sub[col_bmi].values
        gs   = gstar_array(bmis)
        cov  = coverage_curve(gs)
        idx  = np.where(cov >= TARGET_COVER)[0]
        t_cov  = weeks[idx[0]] if len(idx) else weeks[-1]
        cov_at = cov[idx[0]] if len(idx) else cov[-1]
        pc[g] = (weeks, cov, t_cov, cov_at)
    return pc

plot_curves = _build_plot_curves_if_needed()


rows = []
for g, (w, cov, t_cov, cov_at) in plot_curves.items():
    sub = df[df["bmi_group"].astype(str) == g]
    if len(sub) == 0:
        continue
    n = int(len(sub))
    bmi_med = float(sub[col_bmi].median())
    bmi_lo  = float(sub[col_bmi].min())
    bmi_hi  = float(sub[col_bmi].max())
    rows.append({
        "bmi_group": g,
        "bmi_interval": f"{bmi_lo:.1f}~{bmi_hi:.1f}",
        "sample_size": n,
        "median_bmi": round(bmi_med, 2),
        "recommended_week": round(float(t_cov), 1),
        "coverage_at_recommended_week": round(float(cov_at), 3)
    })

rec_table = pd.DataFrame(rows, columns=[
    "bmi_group","bmi_interval","sample_size","median_bmi","recommended_week","coverage_at_recommended_week"
]).sort_values("recommended_week")

# sensitivity details（bmi_group × CV × sigma_multiplier → Recommended week）
CV_GRID     = globals().get("CV_GRID",     [0.05, 0.10, 0.15])
SIGMA_SCALE = globals().get("SIGMA_SCALE", [0.8, 1.0, 1.2])

def _t_cov_for_group(sub_df, cv, sc):
    bmis = sub_df[col_bmi].values
    gs   = gstar_array(bmis, cv=cv, sigma_scale=sc)
    cov  = coverage_curve(gs)
    idx  = np.where(cov >= TARGET_COVER)[0]
    return float(weeks[idx[0]] if len(idx) else weeks[-1])


label_order = globals().get("label_order", list(plot_curves.keys()))

sens_long_rows = []
for g in label_order:
    sub = df[df["bmi_group"].astype(str) == g]
    if len(sub) < 5:
        continue
    for sc in SIGMA_SCALE:
        for cv in CV_GRID:
            t_cov = _t_cov_for_group(sub, cv=cv, sc=sc)
            sens_long_rows.append({
                "bmi_group": g,
                "CV": cv,
                "sigma_multiplier": sc,
                "Recommended week(coverage_at_least_{:.0%})".format(TARGET_COVER): round(t_cov, 1)
            })

sens_long = pd.DataFrame(sens_long_rows, columns=[
    "bmi_group","CV","sigma_multiplier","Recommended week(coverage_at_least_{:.0%})".format(TARGET_COVER)
])


def _maybe_df(name):
    try:
        return globals()[name].copy()
    except KeyError:
        return pd.DataFrame({"Note":[f"Not generated {name}（run the relevant analysis first）"]})

sens_sheet = _maybe_df("sens_df")
boot_sheet = _maybe_df("boot_df")


export_dir  = OUT_DIR if "OUT_DIR" in globals() else str(OUTPUT_DIR)
export_path = os.path.join(export_dir, "problem-2-timing-and-sensitivity.xlsx")

with pd.ExcelWriter(export_path, engine="openpyxl") as writer:
    rec_table.to_excel(writer,   sheet_name="recommendation_summary", index=False)
    sens_long.to_excel(writer,   sheet_name="sensitivity_long",     index=False)
    sens_sheet.to_excel(writer,  sheet_name="sensitivity_matrix",     index=False)
    boot_sheet.to_excel(writer,  sheet_name="BootstrapCI",         index=False)

print(f"\nExported：{export_path}")
print("recommendation summary preview：")
print(rec_table.to_string(index=False))

print("\nsensitivity details preview (first 10 rows)：")
print(sens_long.head(10).to_string(index=False))


## Sensitivity heatmap


In [ ]:


import os, glob, re
import numpy as np
import pandas as pd
import matplotlib, matplotlib.pyplot as plt
from matplotlib import font_manager as fm

# Font configuration
def setup_plot_font(prefer=("DejaVu Sans",)):
    installed = {f.name for f in fm.fontManager.ttflist}
    for fam in prefer:
        if fam in installed:
            matplotlib.rcParams["font.family"] = "sans-serif"
            matplotlib.rcParams["font.sans-serif"] = [fam]
            matplotlib.rcParams["axes.unicode_minus"] = False
            matplotlib.rcParams["pdf.fonttype"] = 42
            matplotlib.rcParams["ps.fonttype"]  = 42
            print(f"[OK] Using system font：{fam}")
            return
    for p in [r"", r"", r""]:
        if os.path.exists(p):
            fm.fontManager.addfont(p)
            fam = fm.FontProperties(fname=p).get_name()
            matplotlib.rcParams["font.family"] = "sans-serif"
            matplotlib.rcParams["font.sans-serif"] = [fam]
            matplotlib.rcParams["axes.unicode_minus"] = False
            matplotlib.rcParams["pdf.fonttype"] = 42
            matplotlib.rcParams["ps.fonttype"]  = 42
            print(f"[OK] Registered font：{p} -> {fam}")
            return
    print("[WARN] No preferred font found; using the default font")
setup_plot_font()


need_build = any(n not in globals() for n in ["df","label_order","col_bmi","weeks","TARGET_COVER","gstar_array","coverage_curve"])
if need_build:
    def safe_read_excel(candidates=None, sheet_candidates=None, engine="openpyxl"):
        cand = (candidates or []) + [
            globals().get("IN_PATH", None),
            r"data/nipt_data.xlsx",
            r"data/nipt_data.xlsx",
            r"data/nipt_data.xlsx",
            "nipt_data.xlsx",
        ]
        cand += glob.glob("**/nipt_data.xlsx", recursive=True)
        cand += glob.glob("**/nipt_data.xlsx", recursive=True)
        fpath, tried = None, []
        for p in cand:
            if not p: 
                continue
            if os.path.exists(p):
                fpath = p; break
            tried.append(os.path.abspath(p))
        if fpath is None:
            raise FileNotFoundError("Data file not found\nTried：\n" + "\n".join("  - "+t for t in tried))
        xls = pd.ExcelFile(fpath, engine=engine)
        sheets = [s.strip() for s in xls.sheet_names]
        sheet_candidates = (sheet_candidates or ["male_data","female_data","Sheet1","Sheet","data"])
        pick = None
        for s in sheet_candidates:
            for real in sheets:
                if s == real or s in real:
                    pick = real; break
            if pick: break
        if pick is None:
            raise ValueError(f"Target worksheet not found; available sheets：{sheets}")
        print(f"[OK] Loaded：{os.path.abspath(fpath)}  |  Sheet：{pick}")
        return pd.read_excel(fpath, sheet_name=pick)

    raw = safe_read_excel()

    def find_col(cols, keys):
        cols = [str(c).strip() for c in cols]
        for k in keys:
            for c in cols:
                if k in c:
                    return c
        return None

    col_bmi   = find_col(raw.columns, ["maternal_bmi","BMI"])
    col_ga    = find_col(raw.columns, ["gestational_age","gestational_age"])
    col_yconc = find_col(raw.columns, ["y_chromosome_fraction","y_chromosome_fraction","y_chromosome_fraction"])

    def parse_week(x):
        if pd.isna(x): return np.nan
        s = str(x).strip()
        m = re.findall(r"(\d+)\s*(?:w|week)\s*\+?\s*(\d+)?", s, flags=re.I)
        if m:
            w = int(m[0][0]); d = int(m[0][1]) if m[0][1] not in [None,""] else 0
            return w + d/7.0
        try: return float(s)
        except: return np.nan

    df = raw[[col_bmi, col_ga, col_yconc]].copy()
    df[col_bmi]   = pd.to_numeric(df[col_bmi], errors="coerce")
    df[col_ga]    = df[col_ga].apply(parse_week)
    df[col_yconc] = pd.to_numeric(df[col_yconc], errors="coerce")
    df = df.dropna(subset=[col_bmi,col_ga,col_yconc]).reset_index(drop=True)

    weeks = np.arange(10.0, 26.0+1e-9, 0.1)
    TARGET_COVER = 0.60
    ASSAY_CV, Z_LOW = 0.10, 1.645

    try:
        from xgboost import XGBRegressor
        reg = XGBRegressor(
            n_estimators=400, learning_rate=0.05, max_depth=4,
            subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
            objective="reg:squarederror", tree_method="hist",
            random_state=42, n_jobs=1
        ).fit(df[[col_ga,col_bmi]].astype(float), df[col_yconc].astype(float))
    except Exception:
        from sklearn.ensemble import RandomForestRegressor
        reg = RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=1)\
              .fit(df[[col_ga,col_bmi]].astype(float), df[col_yconc].astype(float))

    yhat  = reg.predict(df[[col_ga,col_bmi]])
    sigma = float(np.std(df[col_yconc] - yhat, ddof=1))

    bins   = [24, 28, 30, 32, 34, 36, np.inf]
    labels = ["24–28","28–30","30–32","32–34","34–36","≥36"]
    df["bmi_group"] = pd.cut(df[col_bmi], bins=bins, labels=labels, right=False)
    label_order = [lab for lab in labels if lab in df["bmi_group"].dropna().unique()]

    def gstar_array(bmis, cv=ASSAY_CV, sigma_scale=1.0):
        bmis = np.asarray(bmis, dtype=float)
        T, M = len(weeks), len(bmis)
        GA  = np.tile(weeks[:,None], (1, M))
        BMI = np.tile(bmis[None,:], (T, 1))
        Xp  = pd.DataFrame({col_ga: GA.ravel(), col_bmi: BMI.ravel()})
        mu  = reg.predict(Xp).reshape(T, M)
        lo  = mu - Z_LOW * sigma * sigma_scale
        thr = 0.04 * (1 + cv)
        mask = lo >= thr
        has  = mask.any(axis=0)
        first_idx = np.where(has, mask.argmax(axis=0), -1)
        return np.where(has, weeks[first_idx], np.nan)

    def coverage_curve(gs):
        return np.array([float(np.nanmean(gs <= t)) for t in weeks])


CV_GRID     = [0.05, 0.10, 0.15]
SIGMA_SCALE = [0.8, 1.0, 1.2]


def _t_cov_for_group(sub_df, cv, sc):
    bmis = sub_df[col_bmi].values
    gs   = gstar_array(bmis, cv=cv, sigma_scale=sc)
    cov  = coverage_curve(gs)
    idx  = np.where(cov >= TARGET_COVER)[0]
    return weeks[idx[0]] if len(idx) else weeks[-1]

mats, glob_min, glob_max = {}, 1e9, -1e9
for g in label_order:
    sub = df[df["bmi_group"]==g]
    if len(sub) < 5:
        continue
    M = np.zeros((len(SIGMA_SCALE), len(CV_GRID)), dtype=float)
    for i, sc in enumerate(SIGMA_SCALE):
        for j, cv in enumerate(CV_GRID):
            M[i, j] = _t_cov_for_group(sub, cv=cv, sc=sc)
    mats[g] = M
    glob_min = min(glob_min, float(np.nanmin(M)))
    glob_max = max(glob_max, float(np.nanmax(M)))


n = len(mats)
ncols = min(3, n if n>0 else 1)
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3.6*nrows), constrained_layout=True)
if nrows*ncols == 1:
    axes = np.array([[axes]])
elif nrows == 1:
    axes = np.array([axes])

last_im, k = None, 0
for g in label_order:
    if g not in mats: 
        continue
    M = mats[g]
    r, c = divmod(k, ncols); k += 1
    ax = axes[r, c]
    last_im = ax.imshow(M, vmin=glob_min, vmax=glob_max, cmap="viridis", origin="upper")
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f"{M[i,j]:.1f}", ha="center", va="center", color="w", fontsize=10)
    ax.set_xticks(range(len(CV_GRID)));     ax.set_xticklabels([f"{int(100*c)}%" for c in CV_GRID])
    ax.set_yticks(range(len(SIGMA_SCALE))); ax.set_yticklabels([f"×{s:.1f}" for s in SIGMA_SCALE])
    ax.set_xlabel("Assay CV"); ax.set_ylabel("Residual sigma multiplier")
    ax.set_title(f"{g}：Recommended week")

for idx in range(k, nrows*ncols):
    r, c = divmod(idx, ncols)
    axes[r, c].axis("off")

if last_im is not None:
    cbar = fig.colorbar(last_im, ax=axes, shrink=0.92)
    cbar.set_label("Recommended week (first week reaching target coverage)")

plt.suptitle("Sensitivity of recommended week to CV and residual sigma", y=1.02, fontsize=13)
plt.show()


## ROC analysis


In [ ]:


import os, glob, re
import numpy as np
import pandas as pd
import matplotlib, matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, roc_auc_score
from scipy.stats import norm

# Font configuration
def _setup_plot_font(prefer=("DejaVu Sans",)):
    installed = {f.name for f in fm.fontManager.ttflist}
    for fam in prefer:
        if fam in installed:
            matplotlib.rcParams["font.family"] = "sans-serif"
            matplotlib.rcParams["font.sans-serif"] = [fam]
            matplotlib.rcParams["axes.unicode_minus"] = False
            matplotlib.rcParams["pdf.fonttype"] = 42
            matplotlib.rcParams["ps.fonttype"]  = 42
            return
    for p in [r"", r"", r""]:
        if os.path.exists(p):
            fm.fontManager.addfont(p)
            fam = fm.FontProperties(fname=p).get_name()
            matplotlib.rcParams["font.family"] = "sans-serif"
            matplotlib.rcParams["font.sans-serif"] = [fam]
            matplotlib.rcParams["axes.unicode_minus"] = False
            matplotlib.rcParams["pdf.fonttype"] = 42
            matplotlib.rcParams["ps.fonttype"]  = 42
            return
_setup_plot_font()


def _safe_read_excel(candidates=None, sheet_candidates=None, engine="openpyxl"):
    cand = (candidates or []) + [r"nipt_data.xlsx", r"nipt_data.xlsx"]
    cand += glob.glob("**/nipt_data.xlsx", recursive=True)
    cand += glob.glob("**/nipt_data.xlsx", recursive=True)
    fpath = None
    for p in cand:
        if p and os.path.exists(p):
            fpath = p; break
    if fpath is None:
        raise FileNotFoundError("Data file not found; place the authorized workbook at data/nipt_data.xlsx")
    xls = pd.ExcelFile(fpath, engine=engine)
    sheet = xls.sheet_names[0]
    print(f"[OK] Loaded: {os.path.abspath(fpath)} | Sheet: {sheet}")
    return pd.read_excel(fpath, sheet_name=sheet)

def _find_col(cols, keys):
    cols = [str(c).strip() for c in cols]
    for k in keys:
        for c in cols:
            if k in c: return c
    return None

def _parse_week(x):
    if pd.isna(x): return np.nan
    s = str(x).strip()
    m = re.findall(r"(\d+)\s*(?:w|week)\s*\+?\s*(\d+)?", s, flags=re.I)
    if m:
        w = int(m[0][0]); d = int(m[0][1]) if m[0][1] not in [None,""] else 0
        return w + d/7.0
    try: return float(s)
    except: return np.nan

if "df" not in globals():
    raw = _safe_read_excel()
    col_bmi   = _find_col(raw.columns, ["maternal_bmi","BMI"])
    col_ga    = _find_col(raw.columns, ["gestational_age","gestational_age"])
    col_yconc = _find_col(raw.columns, ["y_chromosome_fraction","y_chromosome_fraction","y_chromosome_fraction"])
    if not all([col_bmi, col_ga, col_yconc]):
        raise RuntimeError("Missing columns：BMI/gestational_age/y_chromosome_fraction")
    df = raw[[col_bmi, col_ga, col_yconc]].copy()
    df[col_bmi]   = pd.to_numeric(df[col_bmi], errors="coerce")
    df[col_ga]    = df[col_ga].apply(_parse_week)
    df[col_yconc] = pd.to_numeric(df[col_yconc], errors="coerce")
    df = df.dropna(subset=[col_bmi,col_ga,col_yconc]).reset_index(drop=True)
else:
    if "col_bmi"   not in globals(): col_bmi   = _find_col(df.columns, ["maternal_bmi","BMI"])
    if "col_ga"    not in globals(): col_ga    = _find_col(df.columns, ["gestational_age","gestational_age"])
    if "col_yconc" not in globals(): col_yconc = _find_col(df.columns, ["y_chromosome_fraction","y_chromosome_fraction","y_chromosome_fraction"])


ASSAY_CV = 0.10
THR_RAW  = 0.04
thr_eff  = THR_RAW * (1 + ASSAY_CV)

X_all = df[[col_ga, col_bmi]].astype(float).values
y_all = df[col_yconc].astype(float).values
y_true = (y_all >= thr_eff).astype(int)


if y_true.mean() in [0.0, 1.0]:
    raise RuntimeError("Only one observed class is present; adjust the threshold or inspect the data before plotting ROC")

X_tr, X_te, y_tr_true, y_te_true, y_tr_cont, y_te_cont = train_test_split(
    X_all, y_true, y_all, test_size=0.2, random_state=42, stratify=y_true
)


if "reg" in globals():
    reg_local = reg
else:
    try:
        from xgboost import XGBRegressor
        reg_local = XGBRegressor(
            n_estimators=400, learning_rate=0.05, max_depth=4,
            subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
            objective="reg:squarederror", tree_method="hist",
            random_state=42, n_jobs=1
        ).fit(X_tr, y_tr_cont)
    except Exception:
        from sklearn.ensemble import RandomForestRegressor
        reg_local = RandomForestRegressor(
            n_estimators=400, max_depth=None, random_state=42, n_jobs=1
        ).fit(X_tr, y_tr_cont)


mu_tr = reg_local.predict(X_tr)
sigma_hat = float(np.std(y_tr_cont - mu_tr, ddof=1))
if sigma_hat <= 1e-8:
    sigma_hat = 1e-4


mu_te = reg_local.predict(X_te)
z_te  = (mu_te - thr_eff) / sigma_hat
y_score = norm.cdf(z_te)
y_true  = y_te_true

# ROC
fpr, tpr, thr = roc_curve(y_true, y_score)
auc = roc_auc_score(y_true, y_score)


youden_idx = np.argmax(tpr - fpr)
best_fpr, best_tpr, best_thr = fpr[youden_idx], tpr[youden_idx], thr[youden_idx]

plt.figure(figsize=(6.6, 5.2))
plt.plot(fpr, tpr, linewidth=2, label=f"AUC = {auc:.3f}")
plt.plot([0,1], [0,1], linestyle="--", label="Random classifier")
plt.scatter([best_fpr], [best_tpr], s=36, zorder=5)
plt.annotate(f"Optimal threshold≈{best_thr:.2f}\nTPR={best_tpr:.2f}  FPR={best_fpr:.2f}",
             xy=(best_fpr, best_tpr), xytext=(10, -10),
             textcoords="offset points", fontsize=10,
             bbox=dict(boxstyle="round,pad=0.18", fc="white", ec="#888", lw=0.8, alpha=0.9))
plt.xlabel("False positive rate (FPR)")
plt.ylabel("True positive rate (TPR)")
plt.title("Problem 2 ROC: probability of Y fraction >= 4%")
plt.legend(loc="lower right", frameon=False)
plt.tight_layout()
plt.show()

print(f"AUC = {auc:.3f} | Youden-optimal threshold≈{best_thr:.3f} | TPR={best_tpr:.3f} | FPR={best_fpr:.3f}")


## Measurement-error sensitivity


In [ ]:


import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


CV_GRID     = [0.05, 0.10, 0.15]
SIGMA_SCALE = [0.8, 1.0, 1.2]


def _t_cov_for_group(sub_df, cv, sc):
    bmis = sub_df[col_bmi].values
    gs   = gstar_array(bmis, cv=cv, sigma_scale=sc)
    cov  = coverage_curve(gs)
    idx  = np.where(cov >= TARGET_COVER)[0]
    return float(weeks[idx[0]] if len(idx) else weeks[-1])


rows = []
for g in label_order:
    sub = df[df["bmi_group"] == g]
    if len(sub) < 5:
        continue
    for sc in SIGMA_SCALE:
        for cv in CV_GRID:
            rows.append({
                "bmi_group": g,
                "assay_cv": cv,
                "residual_sigma_multiplier": sc,
                "recommended_week": round(_t_cov_for_group(sub, cv, sc), 1)
            })
sens_table = pd.DataFrame(rows)


export_path = os.path.join(str(OUTPUT_DIR), "problem-2-measurement-error-sensitivity.xlsx")
sens_table.to_excel(export_path, index=False)
print(f"Exported：{export_path}")
print(sens_table.head(10).to_string(index=False))


g = "≥36"
if "bmi_group" not in df.columns:
    raise RuntimeError("Missing bmi_group; create BMI groups first")
if g not in set(df["bmi_group"].astype(str).unique()):

    avail = [str(x) for x in label_order if str(x) in set(df["bmi_group"].astype(str).unique())]
    g = avail[-1] if avail else str(df["bmi_group"].dropna().astype(str).unique()[-1])

sub = df[df["bmi_group"].astype(str) == g]
M = np.zeros((len(SIGMA_SCALE), len(CV_GRID)), dtype=float)
for i, sc in enumerate(SIGMA_SCALE):
    for j, cv in enumerate(CV_GRID):
        M[i, j] = _t_cov_for_group(sub, cv, sc)

plt.figure(figsize=(5, 4))
im = plt.imshow(M, cmap="viridis", origin="upper")
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        plt.text(j, i, f"{M[i,j]:.1f}", ha="center", va="center", color="w")
plt.xticks(range(len(CV_GRID)), [f"{int(c*100)}%" for c in CV_GRID])
plt.yticks(range(len(SIGMA_SCALE)), [f"×{s}" for s in SIGMA_SCALE])
plt.xlabel("Assay CV")
plt.ylabel("Residual sigma multiplier")
plt.title(f"{g} group：Recommended week（error sensitivity）")
plt.colorbar(im, label="Recommended week")
plt.tight_layout()
plt.show()
